In [1]:
import random
import torch
from d2l import torch as d2l

#### 3.3.1 Generating the Dataset

In [2]:
class SyntheticRegressionData(d2l.DataModule):  #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1) * noise
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise 

In [3]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

In [4]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: tensor([ 0.6081, -0.2851]) 
label: tensor([6.3916])


#### 3.3.2 Reading the Dataset

In [5]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train+self.num_val))
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i+self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

In [6]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


#### 3.3.3 Concise Implementation of the Data Loader

In [14]:
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

In [12]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


In [13]:
len(data.train_dataloader())

TypeError: object of type 'generator' has no len()

##### exercise_1
+ The last batch will be smaller than expected.
  In the case that generally, samples are adequate, we may simply ignore the last smaller batch.
+ By PyTorch, adding `drop_last=True` by:
  ~~~ Pytorch
    dataloader = DataLoader(dataset batch_size, drop_last=True)
  ~~~

##### exercise_2
1. Computer will be stuck or the dataset will be incomplete.
2. By designing the seed of random permutation generators, we generate a pseudorandom permutation which
   is not a ture random permutation but could shuffle the dataset. Code may be as follows.
   ``` python
       import random
       random.seed(21)  # 设置种子，结果具有可重复性
       dataset_exe = [1, 2, 3, 4, 5, 6]
       random.shuffle(dataset_exe)
       perint(dataset_exe)
    ```

In [15]:
##### exercise_3
import random
def random_data_generator():
    while True:
        yield random.random()
random_generator = random_data_generator()

print(next(random_generator))
print(next(random_generator))
print(next(random_generator))

0.29564083524513696
0.9223494338402527
0.12165038328093392


##### exercise_4
Actually, random numbers generated by computers are not ture random number.  
They are generated by certain algorithom.  
So as is mentioned in exercise_2, by setting the seed of random,   
we could generate the same set of pseudorandom number each time it is called